In [1]:
# Import required libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Aggregate Fluorescence Intensity Data for HQNO and RHL reporter
## Overview
This notebook aggregates single-cell property data from multiple experimental replicates and positions for the SA HQNO and RHL reporter in co-culture with PA

## Workflow
1. **Input**: Data comes from  `SAReporterGradient` study component in BioImageArchive repo
2. **Data Collection**: Iterates over all replicates and positions and loads `single_cell_props.csv` files.
3. **Output**: Saves the combined dataframe as `hqno_gradient_microscopy.csv` and `rhl_gradient_microscopy.csv`

## Note
Code for further processing of data, model fitting, and figure plotting is located in accompanying [Github repository](https://github.com/simonvanvliet/SpatialToleranceModel)

In [2]:
# Define input and output paths
# set path to BioImageArchive data directory:
data_archive_path = Path('/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2026/BioImageArchive/')

# relative paths to data
input_dir_h =  data_archive_path / 'SAReporterGradient' / 'S11_wt_HQNO-reporter'  
output_dir_h = 'hqno_gradient_microscopy.csv'

input_dir_r = data_archive_path / 'SAReporterGradient' / 'S11_wt_RHL-reporter'
output_dir_r = 'rhl_gradient_microscopy.csv'

process_list = [
    ('HQNO',input_dir_h, output_dir_h),
    ('RHL', input_dir_r, output_dir_r)
]

In [3]:
for type, input_dir, output_dir in process_list:

    all_data = []  # List to store individual dataframes

    if not os.path.isdir(input_dir):
        raise FileNotFoundError(
            f"Could not find input directory: {input_dir}. Current working dir: {os.getcwd()}"
        )


    # Iterate through replicate folders
    for rep_id, replicate_folder in enumerate(os.listdir(input_dir)):
        replicate_path = os.path.join(input_dir, replicate_folder)

        # Process only replicate directories
        if os.path.isdir(replicate_path) and replicate_folder.startswith('replicate'):

            # Iterate through position folders within each replicate
            for pos_folder in os.listdir(replicate_path):
                if pos_folder.startswith('pos'):
                    pos_path = os.path.join(replicate_path, pos_folder, 'single_cell_props.csv')

                    # Load data if CSV file exists
                    if os.path.isfile(pos_path):
                        df = pd.read_csv(pos_path)

                        # Add metadata columns
                        df.insert(1, type, 'wt')
                        df.insert(2, 'replicate', rep_id)
                        df.insert(3, 'pos', pos_folder)
                        df.insert(4, 'replicate_name', replicate_folder)

                        # Append to collection
                        all_data.append(df)

    # Combine all dataframes and save to output file
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        combined_df.to_csv(output_dir, index=False)
        print(f"Successfully combined {len(all_data)} datasets")
        print(f"Total rows: {len(combined_df)}")
        print(f"Output saved to: {output_dir}")

        # Display first few rows
        print(combined_df.columns)
        print(combined_df['replicate_name'].unique())
        print(combined_df['replicate'].unique())

    else:
        print("Warning: No data files found")

Successfully combined 21 datasets
Total rows: 85892
Output saved to: hqno_gradient_microscopy.csv
Index(['label', 'HQNO', 'replicate', 'pos', 'replicate_name', 'area',
       'min_row', 'min_col', 'max_row', 'max_col', 'intensity_max',
       'intensity_mean', 'intensity_min', 'minor_axis_length',
       'major_axis_length', 'y', 'x', 'intensity_gfp', 'intensity_raw_gfp',
       'frame_number', 'intensity_mcherry', 'intensity_raw_mcherry'],
      dtype='str')
<ArrowStringArray>
['replicate_5', 'replicate_7']
Length: 2, dtype: str
[0 1]
Successfully combined 21 datasets
Total rows: 88354
Output saved to: rhl_gradient_microscopy.csv
Index(['label', 'RHL', 'replicate', 'pos', 'replicate_name', 'area', 'min_row',
       'min_col', 'max_row', 'max_col', 'intensity_max', 'intensity_mean',
       'intensity_min', 'minor_axis_length', 'major_axis_length', 'y', 'x',
       'intensity_mcherry', 'intensity_raw_mcherry', 'frame_number'],
      dtype='str')
<ArrowStringArray>
['replicate_3', 'repli